In [1]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from collections import Counter
import math
from fpdf import FPDF
import warnings
warnings.filterwarnings('ignore')

In [2]:
iris = load_iris()
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df['target'] = iris.target
df['target_name'] = df['target'].map({i: name for i, name in enumerate(iris.target_names)})

In [3]:
np.random.seed(42)
for col in df.columns[:-2]:  # skip target columns
    missing_idx = np.random.choice(df.index, size=5, replace=False)
    df.loc[missing_idx, col] = np.nan

print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

Dataset shape: (150, 6)
Columns: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)', 'target', 'target_name']


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target,target_name
0,5.1,3.5,1.4,0.2,0,setosa
1,4.9,3.0,1.4,0.2,0,setosa
2,4.7,3.2,1.3,0.2,0,setosa
3,4.6,3.1,1.5,0.2,0,setosa
4,5.0,3.6,1.4,0.2,0,setosa


In [4]:
data_info = pd.DataFrame({
    'Data Type': df.dtypes,
    'Missing Values': df.isnull().sum(),
    'Missing %': (df.isnull().sum() / len(df)) * 100
})
print("=== Data Types & Missing Values ===")
print(data_info)

=== Data Types & Missing Values ===
                  Data Type  Missing Values  Missing %
sepal length (cm)   float64               5   3.333333
sepal width (cm)    float64               5   3.333333
petal length (cm)   float64               5   3.333333
petal width (cm)    float64               5   3.333333
target                int32               0   0.000000
target_name          object               0   0.000000


In [5]:
print("\n=== Summary Statistics (Numerical) ===")
print(df.describe())


=== Summary Statistics (Numerical) ===
       sepal length (cm)  sepal width (cm)  petal length (cm)  \
count         145.000000        145.000000         145.000000   
mean            5.822069          3.055172           3.739310   
std             0.823397          0.442206           1.766757   
min             4.300000          2.000000           1.000000   
25%             5.100000          2.800000           1.500000   
50%             5.800000          3.000000           4.300000   
75%             6.400000          3.300000           5.100000   
max             7.900000          4.400000           6.900000   

       petal width (cm)      target  
count         145.00000  150.000000  
mean            1.18069    1.000000  
std             0.76778    0.819232  
min             0.10000    0.000000  
25%             0.30000    0.000000  
50%             1.30000    1.000000  
75%             1.80000    2.000000  
max             2.50000    2.000000  


In [6]:
print("\n=== Target Distribution ===")
target_counts = df['target_name'].value_counts()
print(target_counts)


=== Target Distribution ===
target_name
setosa        50
versicolor    50
virginica     50
Name: count, dtype: int64


In [7]:
def entropy(y):
    """Calculate entropy of a target variable."""
    counts = Counter(y)
    probs = [count/len(y) for count in counts.values()]
    return -sum(p * math.log2(p) for p in probs if p > 0)

def information_gain(data, feature, target):
    """
    Calculate information gain of a feature (categorical or discretized).
    data: DataFrame
    feature: column name
    target: column name (categorical)
    """
    # Remove rows where feature is NaN for this calculation
    clean_data = data[[feature, target]].dropna()
    if clean_data.empty:
        return 0.0
    
    parent_entropy = entropy(clean_data[target])
    
    # Weighted entropy after split
    values = clean_data[feature].unique()
    weighted_entropy = 0.0
    for val in values:
        subset = clean_data[clean_data[feature] == val]
        weight = len(subset) / len(clean_data)
        weighted_entropy += weight * entropy(subset[target])
    
    return parent_entropy - weighted_entropy

# For numeric features, we discretize into bins
def discretize_feature(data, feature, bins=3):
    """Convert numeric feature into categorical bins."""
    series = data[feature].dropna()
    if series.nunique() <= bins:
        return data[feature]
    else:
        # Use quantile bins to have roughly equal frequency
        labels = [f"{feature}_bin{i+1}" for i in range(bins)]
        return pd.cut(data[feature], bins=bins, labels=labels, duplicates='drop')


In [8]:
target_series = df['target_name'].dropna()
target_entropy = entropy(target_series)
print(f"Entropy of target variable: {target_entropy:.4f}")

# Information gain for each feature
features = [col for col in df.columns if col not in ['target', 'target_name']]
info_gains = {}

for feat in features:
    # Check if feature is numeric
    if pd.api.types.is_numeric_dtype(df[feat]):
        # Discretize
        discretized = discretize_feature(df, feat, bins=3)
        temp_df = df.copy()
        temp_df[f'{feat}_disc'] = discretized
        gain = information_gain(temp_df, f'{feat}_disc', 'target_name')
    else:
        gain = information_gain(df, feat, 'target_name')
    info_gains[feat] = gain

# Create a DataFrame for easy viewing
info_gain_df = pd.DataFrame(list(info_gains.items()), columns=['Feature', 'Information Gain'])
info_gain_df = info_gain_df.sort_values('Information Gain', ascending=False).reset_index(drop=True)

print("\n=== Information Gain (Feature vs Target) ===")
print(info_gain_df)

Entropy of target variable: 1.5850

=== Information Gain (Feature vs Target) ===
             Feature  Information Gain
0   petal width (cm)          1.423056
1  petal length (cm)          1.319483
2  sepal length (cm)          0.646832
3   sepal width (cm)          0.271441
